# County housing affordability, 2015–2024

This notebook combines monthly Zillow Home Value Index (ZHVI) data, annual Census SAIPE household income estimates, and weekly 30-year mortgage rates to describe changes in county housing affordability.

**Reading guide:** inspect housing coverage → construct annual housing values → load income → validate joins → calculate affordability → build a consistent county panel → compare trends and export tables.

Run the cells from top to bottom. Set `DATA_DIR` below to the folder containing `zhvi.csv`, `mortgage30us.csv`, and `saipe_2015.txt` through `saipe_2024.txt`. Exports are written to `EXPORT_DIR`.

## Scope and interpretation

- The main analysis covers 2015–2024. Housing and rate data for 2025 are inspected separately and do not enter the main panel.
- Each annual housing observation requires all 12 nonmissing monthly values; no housing values are imputed.
- Mortgage payments assume a 20% down payment and a fixed 30-year loan. Payments include principal and interest only; taxes, insurance, maintenance, and other ownership costs are excluded.
- Payment burden is annual modeled mortgage payments divided by annual median household income. A burden of `0.30` means 30% of income.
- Dollar values are nominal. County medians give each included county equal weight and are not population-weighted national household estimates.
- The balanced panel includes counties with complete affordability inputs in every study year. This improves comparisons over time but can exclude counties with weaker data coverage.


## 1. Setup



In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'raw').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
EXPORT_DIR = PROJECT_ROOT / 'data' / 'processed'
if not DATA_DIR.is_dir():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks folder.')
START_YEAR, END_YEAR = 2015, 2024
DOWN_PAYMENT_SHARE = 0.20
LOAN_MONTHS = 30 * 12

plt.rcParams.update({'figure.figsize': (10, 6), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})


## 2. Inspect housing data and monthly coverage

Missing metro labels describe geography; missing monthly ZHVI values determine whether a county-year is eligible. Coverage checks below use available date columns through 2025, so the month count also shows how much of that period the source file contains.


In [ ]:
df = pd.read_csv(DATA_DIR / 'zhvi.csv')
print('Housing shape:', df.shape)
df.head()


In [ ]:
# Review missing values before applying eligibility rules.
# finding the count of missing values per column
missing = df.isna().sum()
missing[missing > 0]

missing_pct = (df.isna().sum()/len(df))*100

missing_pct[missing_pct > 0].sort_values(ascending=False)


In [ ]:
# Identify columns that represent dates

date_columns = df.columns[pd.to_datetime(df.columns, format="mixed", errors="coerce").notna()]
date_columns

# calculate the percentage of counties with an observation in each month

coverage = df[date_columns].notna().mean()*100

coverage_2015 = coverage[pd.to_datetime(coverage.index) >= "2015-01-01"]

coverage_2015.head(12)

coverage_2015.tail(12)


In [ ]:
# Check counties with complete 2015–2025 histories

dates = pd.to_datetime(date_columns)

analysis_columns = date_columns[(dates >= "2015-01-01") & (dates <= "2025-12-31")]

period = df[analysis_columns]

complete_counties = period.notna().all(axis=1).sum()

complete_counties

# percentage of counties with complete histories

complete_pct = complete_counties / len(df) * 100

print(f"Counties with complete data: {complete_counties:,}")
print(f"Percentage of counties: {complete_pct:.2f}%")

missing_months = period.isna().sum(axis=1) # sum of missing months per county

missing_months.describe()

total_months = len(analysis_columns)
total_months

# Create a coverage percentage for each county
county_coverage = ((total_months - missing_months)/total_months)*100
county_coverage.describe()

print("Total months:", total_months)
print("100% coverage:", (county_coverage == 100).sum())
print("At least 99% coverage:", (county_coverage >= 99).sum())
print("At least 95% coverage:", (county_coverage >= 95).sum())
print("At least 90% coverage:", (county_coverage >= 90).sum())
print("Below 90% coverage:", (county_coverage < 90).sum())


## 3. Create county identifiers and annual housing values

Five-digit county FIPS codes preserve leading zeros and provide a stable join key. Reshape monthly columns into county-month records, then take the mean only for county-years with 12 observed months. The mean of monthly ZHVI is an annual index summary, not an average transaction price.


In [ ]:
# Verify geographic identifiers before reshaping.
# Let's check whether our geographic code combination is unique
df.duplicated(
    subset=['StateCodeFIPS', 'MunicipalCodeFIPS']
).sum()

#Create the five character FIPS code
df['county_fips'] = (
    df['StateCodeFIPS'].astype(str).str.zfill(2)
    +
    df['MunicipalCodeFIPS'].astype(str).str.zfill(3)
)

df[
    ['RegionName', 'State', 'StateCodeFIPS',
     'MunicipalCodeFIPS', 'county_fips']
].head(10)

df['county_fips'].str.len().value_counts()


In [ ]:
# Keep missing housing values in the long table so coverage remains measurable.
zhvi_long = df.melt(
    id_vars=[
        'county_fips',
        'RegionName',
        'State'
    ],
    value_vars=analysis_columns,
    var_name='date',
    value_name='zhvi'
)

expected_rows = len(df) * len(analysis_columns)

print("Expected rows:", expected_rows)
print("Actual rows:", len(zhvi_long))

zhvi_long['date'] = pd.to_datetime(
    zhvi_long['date']
)

zhvi_long['year'] = zhvi_long['date'].dt.year

zhvi_long = zhvi_long.rename(
    columns={
        'RegionName': 'county',
        'State': 'state'
    }
)

# Reshape Validation
print("Shape:", zhvi_long.shape)

print(
    "Duplicate county-months:",
    zhvi_long.duplicated(
        subset=['county_fips', 'date']
    ).sum()
)

print(
    "Missing ZHVI:",
    zhvi_long['zhvi'].isna().sum()
)


In [ ]:
# Apply the 12-month requirement before selecting the main study period.
annual_zhvi = (
    zhvi_long
    .groupby(
        ['county_fips', 'county', 'state', 'year'],
        as_index=False
    )
    .agg(
        months_observed=('zhvi', 'count'),
        annual_zhvi=('zhvi', 'mean')
    )
)

annual_zhvi['months_observed'].value_counts().sort_index()

annual_zhvi_complete = annual_zhvi[
    annual_zhvi['months_observed'] == 12
].copy()

housing_main = annual_zhvi_complete[
    annual_zhvi_complete['year'].between(2015, 2024)
].copy()

housing_2025 = annual_zhvi_complete[
    annual_zhvi_complete['year'] == 2025
].copy()

print("Total annual county-years:", len(annual_zhvi))

print(
    annual_zhvi['months_observed']
    .value_counts()
    .sort_index()
)

print(
    "Eligible 2015–2024 county-years:",
    len(housing_main)
)

print(
    "Duplicate FIPS-year keys:",
    housing_main.duplicated(
        subset=['county_fips', 'year']
    ).sum()
)

annual_main = annual_zhvi[
    annual_zhvi['year'].between(2015, 2024)
].copy()

print("2015–2024 total county-years:", len(annual_main))

print(
    "Incomplete 2015–2024 county-years:",
    (annual_main['months_observed'] < 12).sum()
)

print(
    "Complete 2015–2024 county-years:",
    (annual_main['months_observed'] == 12).sum()
)


## 4. Load and validate household income

Use explicit character positions for the fixed-width SAIPE files rather than inferred columns. Exclude state-level records (`county_code == 0`), retain income estimate bounds, and convert nonnumeric income entries to missing values. The bounds are carried forward for sensitivity calculations.


In [ ]:
# A single loader applies the same parsing rules to every year.
def load_saipe(filepath, year):
    """Read county income estimates and bounds from a SAIPE fixed-width file."""

    colspecs = [
        (0, 2),
        (3, 6),
        (133, 139),
        (140, 146),
        (147, 153),
        (193, 238),
        (239, 241)
    ]

    column_names = [
        'state_fips',
        'county_code',
        'median_household_income',
        'income_lower',
        'income_upper',
        'county',
        'state'
    ]

    # Read using Census-defined character positions
    df = pd.read_fwf(
        filepath,
        colspecs=colspecs,
        names=column_names
    )

    # Keep county records only
    df = df[
        df['county_code'] != 0
    ].copy()

    # Convert income fields to numeric
    income_cols = [
        'median_household_income',
        'income_lower',
        'income_upper'
    ]

    df[income_cols] = df[income_cols].apply(
        pd.to_numeric,
        errors='coerce'
    )

    # Construct 5-digit county FIPS
    df['county_fips'] = (
        df['state_fips'].astype(str).str.zfill(2)
        +
        df['county_code'].astype(str).str.zfill(3)
    )

    # Add year
    df['year'] = year

    # Final column order
    df = df[
        [
            'county_fips',
            'county',
            'state',
            'year',
            'median_household_income',
            'income_lower',
            'income_upper'
        ]
    ]

    return df


In [ ]:
saipe_files = {
    year: DATA_DIR / f'saipe_{year}.txt'
    for year in range(START_YEAR, END_YEAR + 1)
}
income = pd.concat(
    [load_saipe(filepath, year) for year, filepath in saipe_files.items()],
    ignore_index=True,
)
income.head()


In [ ]:
# Check join-key uniqueness, missing estimates, and income-bound ordering.
print("Shape:", income.shape)

print("\nYears:")
print(sorted(income['year'].unique()))

print(
    "\nDuplicate FIPS-year keys:",
    income.duplicated(
        subset=['county_fips', 'year']
    ).sum()
)

print("\nMissing income by year:")
print(
    income
    .groupby('year')['median_household_income']
    .apply(lambda x: x.isna().sum())
)

print("\nCounty rows by year:")

print(
    income
    .groupby('year')
    .size()
)

valid_bounds = (
    income['median_household_income'].isna()
    |
    (
        (income['income_lower'] <= income['median_household_income'])
        &
        (income['median_household_income'] <= income['income_upper'])
    )
)

print(
    "Invalid income bounds:",
    (~valid_bounds).sum()
)

invalid_bounds = income[
    income['median_household_income'].notna()
    &
    (
        (income['income_lower'] > income['median_household_income'])
        |
        (income['median_household_income'] > income['income_upper'])
    )
].copy()

print(
    invalid_bounds
    .groupby('year')
    .size()
)


## 5. Match housing with income

Start from eligible housing county-years and use a one-to-one FIPS/year join. Inspect unmatched records before retaining matches. A matched record can still have missing income; those rows are removed when constructing the complete panel.


In [ ]:
# The merge indicator makes exclusions visible.
affordability_base = housing_main.merge(
    income,
    on=['county_fips', 'year'],
    how='left',
    validate='one_to_one',
    indicator=True
)

print("Merged shape:", affordability_base.shape)

print("\nMerge status:")
print(
    affordability_base['_merge']
    .value_counts()
)

unmatched = affordability_base[
    affordability_base['_merge'] == 'left_only'
].copy()

unmatched[
    [
        'county_fips',
        'county_x',
        'state_x',
        'year'
    ]
]


In [ ]:
# Retain housing county/state labels after checking the match status.
affordability_base = affordability_base[
    affordability_base['_merge'] == 'both'
].copy()

affordability_base = affordability_base.drop(columns = ['_merge', 'county_y', 'state_y'])

affordability_base = affordability_base.rename(columns={'county_x': 'county', 'state_x':'state'})
affordability_base.head()

print("Eligible county-years:", len(affordability_base))

print(
    "Duplicate FIPS-year:",
    affordability_base.duplicated(
        subset=['county_fips', 'year']
    ).sum()
)


## 6. Summarize mortgage rates

Calculate a simple average of observed weekly rates for each year. Rates are expressed in percent (for example, 6.0 means 6%), and the observation counts help reveal incomplete annual coverage. The same annual rate is assigned to every county in that year.


In [ ]:
rates = pd.read_csv(DATA_DIR / 'mortgage30us.csv')
rates['MORTGAGE30US'] = pd.to_numeric(rates['MORTGAGE30US'], errors='coerce')
print('Rate shape:', rates.shape)
print('Missing values:')
print(rates.isna().sum())


In [ ]:
rates = rates.rename(columns={'observation_date':'date'})

rates['date'] = pd.to_datetime(rates['date'], errors='coerce')
rates['year'] = rates['date'].dt.year

rates = rates[rates['year'].between(2015, 2025)].copy()

annual_rates = (
    rates
    .groupby(
        'year',
        as_index=False
    )
    .agg(
        weekly_observations=('MORTGAGE30US', 'count'),
        annual_mortgage_rate=('MORTGAGE30US', 'mean')
    )
)

annual_rates


In [ ]:
# Each county-year receives exactly one annual national mortgage rate.
affordability = affordability_base.merge(
    annual_rates[
        ['year', 'annual_mortgage_rate']
    ],
    on='year',
    how='left',
    validate='many_to_one'
)

print("Shape:", affordability.shape)

print(
    "Missing mortgage rates:",
    affordability['annual_mortgage_rate'].isna().sum()
)

print(
    "Duplicate FIPS-year:",
    affordability.duplicated(
        subset=['county_fips', 'year']
    ).sum()
)


## 7. Calculate modeled affordability

For loan principal $L$, monthly interest rate $r$, and $n$ payments, principal and interest is $Lr/[1-(1+r)^{-n}]$. Divide the annual percentage rate by 1,200 to obtain the monthly decimal rate. Price-to-income is a value multiple; payment burden is an income share.


In [ ]:
# Model a purchase at annual ZHVI with the assumptions defined in Setup.
affordability['loan_amount'] = (
    affordability['annual_zhvi'] * (1 - DOWN_PAYMENT_SHARE)
)

affordability['monthly_rate'] = (
    affordability['annual_mortgage_rate'] / 1200
)

affordability['monthly_payment'] = (
    affordability['loan_amount']
    *
    (
        affordability['monthly_rate']
        /
        (
            1
            -
            (1 + affordability['monthly_rate']) ** -LOAN_MONTHS
        )
    )
)

affordability['payment_burden'] = (
    affordability['monthly_payment'] * 12
    /
    affordability['median_household_income']
)

affordability['price_to_income'] = (
    affordability['annual_zhvi']
    /
    affordability['median_household_income']
)

print("Rows:", len(affordability))

print(
    "Missing rates:",
    affordability['annual_mortgage_rate'].isna().sum()
)

print(
    "Missing payments:",
    affordability['monthly_payment'].isna().sum()
)

print(
    "Missing burden:",
    affordability['payment_burden'].isna().sum()
)

affordability[
    [
        'county_fips',
        'county',
        'state',
        'year',
        'annual_zhvi',
        'median_household_income',
        'annual_mortgage_rate',
        'loan_amount',
        'monthly_payment',
        'payment_burden',
        'price_to_income'
    ]
].head()


In [ ]:
# Sanity check: a $320,000 loan at 6% over 30 years costs about $1,918.56 per month.
test_home_value = 400000
test_loan = test_home_value * 0.80

test_annual_rate = 6.0
test_monthly_rate = test_annual_rate / 1200

test_n = 360

test_payment = (
    test_loan
    *
    (
        test_monthly_rate
        /
        (
            1
            -
            (1 + test_monthly_rate) ** -test_n
        )
    )
)

print("Loan:", test_loan)
print("Monthly rate:", test_monthly_rate)
print("Monthly payment:", test_payment)


### Income sensitivity and plausibility checks

Higher income produces lower payment burden, so the denominator bounds reverse: use the upper income estimate for the lower burden and vice versa. These bounds reflect income uncertainty only; they do not account for home-value or mortgage-rate uncertainty.


In [ ]:
affordability['burden_income_lower'] = (
    affordability['monthly_payment'] * 12
    /
    affordability['income_upper']
)

affordability['burden_income_upper'] = (
    affordability['monthly_payment'] * 12
    /
    affordability['income_lower']
)

valid_burden_bounds = (
    affordability['payment_burden'].isna()
    |
    (
        (
            affordability['burden_income_lower']
            <=
            affordability['payment_burden']
        )
        &
        (
            affordability['payment_burden']
            <=
            affordability['burden_income_upper']
        )
    )
)

print(
    "Invalid burden bounds:",
    (~valid_burden_bounds).sum()
)

affordability[
    [
        'annual_zhvi',
        'median_household_income',
        'annual_mortgage_rate',
        'monthly_payment',
        'payment_burden',
        'price_to_income'
    ]
].describe()

print(
    "Nonpositive ZHVI:",
    (affordability['annual_zhvi'] <= 0).sum()
)

print(
    "Nonpositive income:",
    (affordability['median_household_income'] <= 0).sum()
)

print(
    "Nonpositive payments:",
    (affordability['monthly_payment'] <= 0).sum()
)

print(
    "Negative burden:",
    (affordability['payment_burden'] < 0).sum()
)


## 8. Build the balanced county panel

First remove rows missing a required affordability input. Then retain counties observed in every year from 2015 through 2024. This final panel is the common sample used by all comparisons below.


In [ ]:
# Filter complete inputs before counting observed years for each county.
complete_affordability = affordability[
    affordability[
        [
            'annual_zhvi',
            'median_household_income',
            'annual_mortgage_rate',
            'payment_burden'
        ]
    ].notna().all(axis=1)
].copy()

county_coverage = (
    complete_affordability
    .groupby('county_fips')['year']
    .nunique()
    .reset_index(name='years_observed')
)

consistent_counties = county_coverage[
    county_coverage['years_observed'] == (END_YEAR - START_YEAR + 1)
]['county_fips']

main_panel = complete_affordability[
    complete_affordability['county_fips'].isin(
        consistent_counties
    )
].copy()

print(
    "Consistent counties:",
    main_panel['county_fips'].nunique()
)

print(
    "County-year observations:",
    len(main_panel)
)

print(
    "Years:",
    sorted(main_panel['year'].unique())
)

print(
    "Duplicate FIPS-year:",
    main_panel.duplicated(
        subset=['county_fips', 'year']
    ).sum()
)

main_panel.groupby('year')['county_fips'].nunique()


## 9. Summarize annual trends

Each row describes the median county in that year. A change in these annual medians is distinct from the median of individual county changes, which is calculated in the next section. Percentage-point changes apply to burden shares; percentage growth applies to dollar values.


In [ ]:
annual_summary = (
    main_panel
    .groupby('year', as_index=False)
    .agg(
        median_zhvi=('annual_zhvi', 'median'),
        median_income=('median_household_income', 'median'),
        median_monthly_payment=('monthly_payment', 'median'),
        median_payment_burden=('payment_burden', 'median'),
        median_price_to_income=('price_to_income', 'median'),
        mortgage_rate=('annual_mortgage_rate', 'first')
    )
)

annual_summary

annual_summary['median_payment_burden_pct'] = (
    annual_summary['median_payment_burden'] * 100
)

annual_summary[
    [
        'year',
        'median_zhvi',
        'median_income',
        'mortgage_rate',
        'median_monthly_payment',
        'median_payment_burden_pct',
        'median_price_to_income'
    ]
]


In [ ]:
# Compare the first and last study years using annual county medians.
summary_2015 = annual_summary[
    annual_summary['year'] == 2015
].iloc[0]

summary_2024 = annual_summary[
    annual_summary['year'] == 2024
].iloc[0]

burden_change_pp = (
    summary_2024['median_payment_burden_pct']
    -
    summary_2015['median_payment_burden_pct']
)

print(
    "Payment burden change:",
    round(burden_change_pp, 2),
    "percentage points"
)

def percent_change(start, end):
    """Return percentage growth relative to the starting value."""
    return ((end - start)/start)*100

zhvi_growth = percent_change(
    summary_2015['median_zhvi'],
    summary_2024['median_zhvi']
)

income_growth = percent_change(
    summary_2015['median_income'],
    summary_2024['median_income']
)

payment_growth = percent_change(
    summary_2015['median_monthly_payment'],
    summary_2024['median_monthly_payment']
)

print("Median home value growth:", round(zhvi_growth, 1), "%")
print("Median income growth:", round(income_growth, 1), "%")
print("Median mortgage payment growth:", round(payment_growth, 1), "%")


## 10. Compare county changes

Join each county’s 2015 and 2024 records to measure change within the same county. A positive burden change indicates higher modeled principal-and-interest costs relative to income.


In [ ]:
# Pair endpoints using county FIPS; validate that each county appears once.
panel_2015 = main_panel[
    main_panel['year'] == 2015
].copy()

panel_2024 = main_panel[
    main_panel['year'] == 2024
].copy()

panel_2015 = panel_2015[
    [
        'county_fips',
        'county',
        'state',
        'annual_zhvi',
        'median_household_income',
        'monthly_payment',
        'payment_burden',
        'price_to_income'
    ]
]

panel_2024 = panel_2024[
    [
        'county_fips',
        'annual_zhvi',
        'median_household_income',
        'monthly_payment',
        'payment_burden',
        'price_to_income'
    ]
]

county_change = panel_2015.merge(
    panel_2024,
    on='county_fips',
    how='inner',
    validate='one_to_one',
    suffixes=('_2015', '_2024')
)


In [ ]:
# Report both typical county changes and the share with increasing burden.
# Home-value growth relative to 2015.
county_change['zhvi_growth_pct'] = (
    (county_change['annual_zhvi_2024'] - county_change['annual_zhvi_2015'])
    /
    county_change['annual_zhvi_2015']) * 100

# income change
county_change['income_growth_pct'] = (
    (county_change['median_household_income_2024'] - county_change['median_household_income_2015'])
    /
    county_change['median_household_income_2015']) * 100

# monthly payment change
county_change['payment_growth_pct'] = (
    (county_change['monthly_payment_2024'] - county_change['monthly_payment_2015'])
    /
    county_change['monthly_payment_2015']) * 100

# Convert the change in income share to percentage points.
county_change['burden_change_pp'] = (
    county_change['payment_burden_2024']
    -
    county_change['payment_burden_2015']
) * 100

print(
    "Median home-value growth:",
    round(county_change['zhvi_growth_pct'].median(), 1),
    "%"
)

print(
    "Median income growth:",
    round(county_change['income_growth_pct'].median(), 1),
    "%"
)

print(
    "Median mortgage-payment growth:",
    round(county_change['payment_growth_pct'].median(), 1),
    "%"
)

print(
    "Median burden change:",
    round(county_change['burden_change_pp'].median(), 2),
    "percentage points"
)

counties_worse = (
    county_change['burden_change_pp'] > 0
).sum()

total_counties = len(county_change)

share_worse = (
    counties_worse / total_counties
) * 100

print("Counties with higher burden:", counties_worse)
print("Total counties:", total_counties)
print("Share with higher burden:", round(share_worse, 2), "%")


In [ ]:
county_change['burden_change_pp'].describe()

county_change['burden_change_pp'].quantile(
    [0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99]
)


In [ ]:
# The dashed line marks the median within-county burden change.
plt.figure(figsize=(10, 6))

plt.hist(
    county_change['burden_change_pp'],
    bins=40
)

plt.axvline(
    county_change['burden_change_pp'].median(),
    linestyle='--',
    label='Median'
)

plt.xlabel('Change in Payment Burden (Percentage Points)')
plt.ylabel('Number of Counties')
plt.title('Distribution of County Mortgage Payment Burden Changes, 2015–2024')
plt.legend()
plt.tight_layout()

plt.show()


In [ ]:
# Express endpoint burdens as percentages for readable comparison tables.
county_change['payment_burden_2015_pct'] = (
    county_change['payment_burden_2015'] * 100
)

county_change['payment_burden_2024_pct'] = (
    county_change['payment_burden_2024'] * 100
)

largest_increases = (
    county_change
    .sort_values(
        'burden_change_pp',
        ascending=False
    )
    .head(10)
)

largest_increases[
    [
        'county',
        'state',
        'payment_burden_2015_pct',
        'payment_burden_2024_pct',
        'burden_change_pp',
        'zhvi_growth_pct',
        'income_growth_pct'
    ]
].round(2)

smallest_changes = (
    county_change
    .sort_values(
        'burden_change_pp',
        ascending=True
    )
    .head(10)
)

smallest_changes[
    [
        'county',
        'state',
        'payment_burden_2015_pct',
        'payment_burden_2024_pct',
        'burden_change_pp',
        'zhvi_growth_pct',
        'income_growth_pct'
    ]
].round(2)

county_change['price_to_income_change'] = (
    county_change['price_to_income_2024']
    -
    county_change['price_to_income_2015']
)


## 11. Prepare and export presentation tables

Keep full precision in the analysis tables and round only the presentation tables. `kpi_summary` combines annual median burden changes with median within-county growth; those summaries answer different questions. CSV exports overwrite files of the same name in `EXPORT_DIR`.


In [ ]:
kpi_summary = {
    '2015 Median Burden (%)':
        annual_summary.loc[
            annual_summary['year'] == 2015,
            'median_payment_burden_pct'
        ].iloc[0],

    '2024 Median Burden (%)':
        annual_summary.loc[
            annual_summary['year'] == 2024,
            'median_payment_burden_pct'
        ].iloc[0],

    'Burden Change (pp)':
        annual_summary.loc[
            annual_summary['year'] == 2024,
            'median_payment_burden_pct'
        ].iloc[0]
        -
        annual_summary.loc[
            annual_summary['year'] == 2015,
            'median_payment_burden_pct'
        ].iloc[0],

    'Median Home Value Growth (%)':
        county_change['zhvi_growth_pct'].median(),

    'Median Income Growth (%)':
        county_change['income_growth_pct'].median(),

    'Median Mortgage Payment Growth (%)':
        county_change['payment_growth_pct'].median(),

    'Counties with Higher Burden (%)':
        (county_change['burden_change_pp'] > 0).mean() * 100
}

kpi_summary

trend_table = annual_summary[
    [
        'year',
        'median_zhvi',
        'median_income',
        'mortgage_rate',
        'median_monthly_payment',
        'median_payment_burden_pct',
        'median_price_to_income'
    ]
].copy()

trend_table = trend_table.round({
    'median_zhvi': 0,
    'median_income': 0,
    'mortgage_rate': 2,
    'median_monthly_payment': 0,
    'median_payment_burden_pct': 2,
    'median_price_to_income': 2
})

trend_table


In [ ]:
# Prepare county detail, the largest burden increases, and growth comparisons.
county_dashboard = county_change[
    [
        'county_fips',
        'county',
        'state',
        'annual_zhvi_2015',
        'annual_zhvi_2024',
        'median_household_income_2015',
        'median_household_income_2024',
        'payment_burden_2015_pct',
        'payment_burden_2024_pct',
        'zhvi_growth_pct',
        'income_growth_pct',
        'payment_growth_pct',
        'burden_change_pp'
    ]
].copy()

county_dashboard = county_dashboard.round(2)

top_10_deterioration = (
    county_dashboard
    .nlargest(
        10,
        'burden_change_pp'
    )
    [
        [
            'county',
            'state',
            'burden_change_pp',
            'zhvi_growth_pct',
            'income_growth_pct'
        ]
    ]
)

top_10_deterioration

growth_comparison = pd.DataFrame({
    'Metric': [
        'Home Value',
        'Household Income',
        'Mortgage Payment'
    ],
    'Median Growth (%)': [
        county_change['zhvi_growth_pct'].median(),
        county_change['income_growth_pct'].median(),
        county_change['payment_growth_pct'].median()
    ]
})

growth_comparison['Median Growth (%)'] = (
    growth_comparison['Median Growth (%)']
    .round(1)
)

growth_comparison


In [ ]:
# Keep generated files together and show the destination after writing.
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
export_tables = {
    'affordability_trends.csv': trend_table,
    'county_affordability_changes.csv': county_dashboard,
    'growth_comparison.csv': growth_comparison,
    'top_10_deterioration.csv': top_10_deterioration,
}
for filename, table in export_tables.items():
    table.to_csv(EXPORT_DIR / filename, index=False)
print(f'Exported {len(export_tables)} tables to {EXPORT_DIR.resolve()}')


## Reading the results

Interpret these estimates as a consistent purchase-cost scenario for the included counties. They describe associations over time and do not identify causal effects. Existing owners may have different loan terms, and the modeled payment excludes several ownership costs. Consult the coverage, merge, and panel checks above before generalizing to counties outside this sample.
